# 🧠 Understanding VLM Router Architectures

This notebook explores **all 3 router types** trained in the Artemis project:

| Router Type | Training Strategy | Output |
|-------------|------------------|--------|
| **Classical** | Cross-Entropy + KL Divergence | 5-class probabilities |
| **Pairwise** | Margin Ranking Loss | Score per model |
| **Reward** | MSE on Rewards | Predicted reward |

Each router uses **DistilBERT** as the text encoder and predicts the best VLM model for a given query.

## 📦 Setup & Imports

In [ ]:
# === Path Setup ===
import sys
from pathlib import Path

# Notebook paths
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent  # artemis_final/
ROOT_DIR = ARTEMIS_DIR.parent             # Which_VLM_Router/
ROUTER_TRAIN_DIR = ARTEMIS_DIR / 'router_train'

# Add to sys.path for imports
for p in [str(ARTEMIS_DIR), str(ROOT_DIR), str(ROUTER_TRAIN_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 Setup complete")

%load_ext autoreload
%autoreload 2

: 

In [ ]:
import sys
from pathlib import Path

# Add paths
sys.path.insert(0, str(Path.cwd().parent.parent))  # artemis_final/
sys.path.insert(0, str(Path.cwd().parent.parent / 'router_train'))  # router_train/

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✓ Imports successful')

In [ ]:
# Detect best device
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'🚀 Using CUDA: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('🍎 Using Apple MPS')
else:
    DEVICE = 'cpu'
    print('💻 Using CPU')

# Checkpoint paths
CHECKPOINTS_DIR = Path.cwd().parent.parent / 'checkpoints'  # artemis_final/checkpoints/
print(f'\nCheckpoints directory: {CHECKPOINTS_DIR}')
print('Available checkpoints:')
for f in CHECKPOINTS_DIR.glob('*.pt'):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  - {f.name} ({size_mb:.1f} MB)')

---
## 1️⃣ Classical Router (CE + KL Loss)

**Architecture:** Text → DistilBERT → Mode Embedding → MLP → 5-class softmax

**Training:** Cross-Entropy on hard labels + KL divergence on soft labels (teacher distribution)

In [ ]:
from artemis_router.inference_classical_router import ClassicalRouterInference

# Load Classical Router
classical_router = ClassicalRouterInference(
    checkpoint_path=str(CHECKPOINTS_DIR / 'best_classical_router.pt'),
    device=DEVICE,
    verbose=True
)

print('\n📊 Classical Router Stats:')
stats = classical_router.get_stats()
for k, v in stats.items():
    print(f'  {k}: {v}')

In [ ]:
# Test Classical Router with different prompts
test_prompts = [
    ('What is the capital of France?', 'qa', 'accuracy'),
    ('Extract all text from this receipt.', 'ocr', 'fast'),
    ('Analyze the trend in this bar chart.', 'chartqa', 'balanced'),
    ('Explain this scientific diagram in detail.', 'diagram_reasoning', 'accuracy'),
    ('What color is the cat?', 'vqa', 'cheap'),
]

print('🎯 Classical Router Predictions:')
print('=' * 80)

classical_results = []
for prompt, task, mode in test_prompts:
    result = classical_router.route(
        prompt=prompt,
        mode=mode,
        metadata={'router_task': task, 'source_dataset': 'test'}
    )
    classical_results.append(result)
    
    print(f'\n📝 Prompt: "{prompt[:50]}..."')
    print(f'   Task: {task}, Mode: {mode}')
    print(f'   ✓ Chosen: {result["chosen_model"]} ({result["probs"][result["chosen_model"]]:.2%})')
    print(f'   ⏱️ Latency: {result["inference_ms"]:.1f}ms')

In [ ]:
# Visualize probability distributions for Classical Router
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (result, (prompt, task, mode)) in enumerate(zip(classical_results, test_prompts)):
    if idx >= 5:
        break
    ax = axes[idx]
    
    models = list(result['probs'].keys())
    probs = [result['probs'][m] for m in models]
    colors = ['#2ecc71' if m == result['chosen_model'] else '#3498db' for m in models]
    
    bars = ax.barh(models, probs, color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Probability')
    ax.set_title(f'{task} ({mode})', fontsize=10)
    
    # Add value labels
    for bar, prob in zip(bars, probs):
        ax.text(prob + 0.02, bar.get_y() + bar.get_height()/2,
                f'{prob:.2%}', va='center', fontsize=9)

axes[5].axis('off')
fig.suptitle('Classical Router: Probability Distributions per Query', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2️⃣ Pairwise Router (Margin Ranking Loss)

**Architecture:** Text + Model Embedding + Mode Embedding → MLP → Scalar Score

**Training:** Margin ranking loss on (better_model, worse_model) pairs

In [ ]:
from artemis_router.inference_pairwise_router import PairwiseRouterInference

# Load Pairwise Router
pairwise_router = PairwiseRouterInference(
    checkpoint_path=str(CHECKPOINTS_DIR / 'best_pairwise_router.pt'),
    device=DEVICE,
    verbose=True
)

print('\n📊 Pairwise Router Stats:')
stats = pairwise_router.get_stats()
for k, v in stats.items():
    print(f'  {k}: {v}')

In [ ]:
# Test Pairwise Router
print('🎯 Pairwise Router Predictions:')
print('=' * 80)

pairwise_results = []
for prompt, task, mode in test_prompts:
    result = pairwise_router.route(
        prompt=prompt,
        mode=mode,
        metadata={'router_task': task, 'source_dataset': 'test'}
    )
    pairwise_results.append(result)
    
    print(f'\n📝 Prompt: "{prompt[:50]}..."')
    print(f'   Task: {task}, Mode: {mode}')
    print(f'   ✓ Chosen: {result["chosen_model"]} (score: {result["scores"][result["chosen_model"]]:.3f})')
    print(f'   📊 Ranking: {pairwise_router.rank_models(prompt, mode)[0:3]}...')
    print(f'   ⏱️ Latency: {result["inference_ms"]:.1f}ms')

In [ ]:
# Visualize scores for Pairwise Router
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (result, (prompt, task, mode)) in enumerate(zip(pairwise_results, test_prompts)):
    if idx >= 5:
        break
    ax = axes[idx]
    
    models = list(result['scores'].keys())
    scores = [result['scores'][m] for m in models]
    colors = ['#e74c3c' if m == result['chosen_model'] else '#9b59b6' for m in models]
    
    bars = ax.barh(models, scores, color=colors)
    ax.axvline(x=0, color='black', linestyle='--', alpha=0.3)
    ax.set_xlabel('Score')
    ax.set_title(f'{task} ({mode})', fontsize=10)
    
    for bar, score in zip(bars, scores):
        ax.text(score + 0.05 if score >= 0 else score - 0.2, 
                bar.get_y() + bar.get_height()/2,
                f'{score:.2f}', va='center', fontsize=9)

axes[5].axis('off')
fig.suptitle('Pairwise Router: Model Scores per Query', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔄 Comparing Router Decisions

Let's see how the two routers compare on the same queries.

In [ ]:
# Compare decisions
comparison_data = []

for (prompt, task, mode), c_result, p_result in zip(test_prompts, classical_results, pairwise_results):
    comparison_data.append({
        'Task': task,
        'Mode': mode,
        'Classical': c_result['chosen_model'],
        'Classical_Confidence': c_result['probs'][c_result['chosen_model']],
        'Pairwise': p_result['chosen_model'],
        'Pairwise_Score': p_result['scores'][p_result['chosen_model']],
        'Agreement': c_result['chosen_model'] == p_result['chosen_model']
    })

df_comparison = pd.DataFrame(comparison_data)
print('📊 Router Comparison:')
print(df_comparison.to_string(index=False))

agreement_rate = df_comparison['Agreement'].mean()
print(f'\n🤝 Agreement Rate: {agreement_rate:.1%}')

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model selection frequency
classical_counts = df_comparison['Classical'].value_counts()
pairwise_counts = df_comparison['Pairwise'].value_counts()

x = np.arange(len(classical_router.model_names))
width = 0.35

ax1 = axes[0]
c_vals = [classical_counts.get(m, 0) for m in classical_router.model_names]
p_vals = [pairwise_counts.get(m, 0) for m in classical_router.model_names]

ax1.bar(x - width/2, c_vals, width, label='Classical', color='#3498db')
ax1.bar(x + width/2, p_vals, width, label='Pairwise', color='#e74c3c')
ax1.set_xticks(x)
ax1.set_xticklabels(classical_router.model_names, rotation=45, ha='right')
ax1.set_ylabel('Selection Count')
ax1.set_title('Model Selection Frequency')
ax1.legend()

# Latency comparison  
ax2 = axes[1]
classical_latencies = [r['inference_ms'] for r in classical_results]
pairwise_latencies = [r['inference_ms'] for r in pairwise_results]

ax2.boxplot([classical_latencies, pairwise_latencies], labels=['Classical', 'Pairwise'])
ax2.set_ylabel('Latency (ms)')
ax2.set_title('Inference Latency Comparison')

plt.tight_layout()
plt.show()

print(f'\n⏱️ Mean Latency - Classical: {np.mean(classical_latencies):.1f}ms, Pairwise: {np.mean(pairwise_latencies):.1f}ms')

---
## 📝 Summary

| Router | Strengths | Best For |
|--------|-----------|----------|
| **Classical** | Direct probability output, faster | Production deployment |
| **Pairwise** | Explicit rankings, interpretable scores | Analysis & debugging |
| **Reward** | Multi-objective optimization | When quality metrics vary by mode |